### Import Packages

In [200]:
import pandas as pd

import nltk

# sentence tokenizer and word tokenizer
from nltk.tokenize import sent_tokenize, word_tokenize 
# Subword tokenizer using BERT
from transformers import AutoTokenizer          
# Subword tokenizer using GPT  
import tiktoken 
# Subword tokenizer using SentencePiece
import sentencepiece as spm 

# Stopwords
from nltk.corpus import stopwords

# POS - Parts of speech
from nltk import pos_tag

# Tag-Conversion 
from nltk.corpus import wordnet

# Lemmatize
from nltk.stem import WordNetLemmatizer

import string
import re

#### Downloading nltk ( stopwords) packages:

In [201]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jagad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jagad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jagad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jagad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

#### Downloading POS tagger model:

In [202]:
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
print("POS tagger model is downloaded")

POS tagger model is downloaded


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\jagad\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\jagad\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


#### Load the dataset

In [203]:
clinical_trial_original_data = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_5-Clinical_Trial_Disease_Category_Classification\data\preprocessed_data\clinical_dataset_after_cleaning_null_values.csv")


#### Explore the data

In [204]:
# To find the shape of the data:
print("Datashape: ", clinical_trial_original_data.shape)
print("-" * 50)

# To find data type of each fearures:
print("\nInformation: ")
print(clinical_trial_original_data.info())
print("-" * 50)

# To find null values in each count:
print("\nSum of misisng values: ")
print(clinical_trial_original_data.isnull().sum())
print("-" * 50)

Datashape:  (60337, 16)
--------------------------------------------------

Information: 
<class 'pandas.DataFrame'>
RangeIndex: 60337 entries, 0 to 60336
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   source_condition_query  60337 non-null  str    
 1   nct_id                  60337 non-null  str    
 2   title                   60337 non-null  str    
 3   official_title          60337 non-null  str    
 4   brief_summary           60337 non-null  str    
 5   conditions              60337 non-null  str    
 6   interventions           60337 non-null  str    
 7   overall_status          60337 non-null  str    
 8   study_type              60337 non-null  str    
 9   phase                   60337 non-null  str    
 10  sex                     60337 non-null  str    
 11  healthy_volunteers      60337 non-null  bool   
 12  eligibility_criteria    60337 non-null  str    
 13  clinicaltria

### Take a copy of original dataframe and create a new one.

In [ ]:
clinical_trial_df = clinical_trial_original_data.copy()

clinical_trial_df = clinical_trial_df[['source_condition_query', 'brief_summary']]

# To find the shape of the data:
print("Datashape: ", clinical_trial_df.shape)
print("-" * 50)

# To find data type of each fearures:
print("\nInformation: ")
print(clinical_trial_df.info())
print("-" * 50)

# To find null values in each count:
print("\nSum of misisng values: ")
print(clinical_trial_df.isnull().sum())
print("-" * 50)

# To Find the duplicate values:
print("\nNo of duplicate value:" , clinical_trial_df.duplicated().sum())
print("-" * 50)



Datashape:  (60337, 2)
--------------------------------------------------

Information: 
<class 'pandas.DataFrame'>
RangeIndex: 60337 entries, 0 to 60336
Data columns (total 2 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   source_condition_query  60337 non-null  str  
 1   brief_summary           60337 non-null  str  
dtypes: str(2)
memory usage: 942.9 KB
None
--------------------------------------------------

Sum of misisng values: 
source_condition_query    0
brief_summary             0
dtype: int64
--------------------------------------------------

No of duplicate value: 265
--------------------------------------------------


### Drop the Duplicated records:

In [206]:
clinical_df = clinical_trial_df.drop_duplicates() 

In [207]:
# To find the shape of the data:
print("After Droping Duplicates:")
print("\nDatashape: ", clinical_df.shape)
print("-" * 50)

# To find data type of each fearures:
print("\nInformation: ")
print(clinical_df.info())
print("-" * 50)

# To find null values in each count:
print("\nSum of misisng values: ")
print(clinical_df.isnull().sum())
print("-" * 50)

# To Find the duplicate values:
print("\nNo of duplicate value:" , clinical_df.duplicated().sum())
print("-" * 50)

After Droping Duplicates:

Datashape:  (60072, 2)
--------------------------------------------------

Information: 
<class 'pandas.DataFrame'>
Index: 60072 entries, 0 to 60336
Data columns (total 2 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   source_condition_query  60072 non-null  str  
 1   brief_summary           60072 non-null  str  
dtypes: str(2)
memory usage: 1.4 MB
None
--------------------------------------------------

Sum of misisng values: 
source_condition_query    0
brief_summary             0
dtype: int64
--------------------------------------------------

No of duplicate value: 0
--------------------------------------------------


#### sample tokenization

In [ ]:
# Taking first row as sample:
sample_text = clinical_df['brief_summary'].iloc[0]
print("Sample text choosen from the feature:")
print(sample_text)

# Separating the punctuations:
fixed_sample_text = re.sub(r'\.(?=[A-Z])', '. ', sample_text)
print("\nSeparating punctuations:")
print(fixed_sample_text)

# Lower casing:
lower_text = fixed_sample_text.lower()
print("\nAfter converting each word into lower case:")
print(lower_text)

# Tokenization by sentence:
tokens_sentence = sent_tokenize(lower_text)
print("\nTokenization for sentence:")
print(tokens_sentence)

# Tokenization by word:
tokens_word = word_tokenize(lower_text)
print("\nTokenization for word:")
print(tokens_word)

# stop words removal for sample_text:
stop_words_sample_text = set(stopwords.words('english'))

# punctuations removal for sample_text:
punctuations_sample_text = set(string.punctuation)

filtered_tokens = [ 
                    word for word in tokens_word
                    if word not in punctuations_sample_text and word not in stop_words_sample_text
]
print("\nAfter removing stopwords and punctuations from the sentence:")
print(filtered_tokens)

# POS - Parts of speech
tagged = pos_tag(filtered_tokens)
print(tagged)


def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default fallback


# Lemmatization without POS:
# lemmatizer = WordNetLemmatizer()
# lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tagged]
# print("\nAfter lemmatization:")
# print(lemmatized_tokens)


Sample text choosen from the feature:
Breast cancer patients often have perioperative emotional disorders such as anxiety and depression, which can lead to poor quality of recovery.This study aims to determine whether ketamine could improve the quality of recovery in breast cancer patients. Meanwhile, it will show if ketamine could improve anxiety, depression, postoperative pain and fatigue.This trial also will bring great concerns on patients' mental health perioperatively and explore the measures to improve their quality of life.

Separating punctuations:
Breast cancer patients often have perioperative emotional disorders such as anxiety and depression, which can lead to poor quality of recovery. This study aims to determine whether ketamine could improve the quality of recovery in breast cancer patients. Meanwhile, it will show if ketamine could improve anxiety, depression, postoperative pain and fatigue. This trial also will bring great concerns on patients' mental health periopera

##### Lemmatization

In [209]:
lemmatized_tokens = [
                        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
                        for word, tag in tagged
]
print("After POS and lemmatization:")
print(lemmatized_tokens)

After POS and lemmatization:
['breast', 'cancer', 'patient', 'often', 'perioperative', 'emotional', 'disorder', 'anxiety', 'depression', 'lead', 'poor', 'quality', 'recovery', 'study', 'aim', 'determine', 'whether', 'ketamine', 'could', 'improve', 'quality', 'recovery', 'breast', 'cancer', 'patient', 'meanwhile', 'show', 'ketamine', 'could', 'improve', 'anxiety', 'depression', 'postoperative', 'pain', 'fatigue', 'trial', 'also', 'bring', 'great', 'concern', 'patient', 'mental', 'health', 'perioperatively', 'explore', 'measure', 'improve', 'quality', 'life']


### Applying to the feature 'brief_summary:

In [213]:
stop_words = set(stopwords.words('english'))
punctuations = set(string.punctuation)
lemmatizer = WordNetLemmatizer()


def cleaned_text(text):

    def get_wordnet_pos(treebank_tag):
        if treebank_tag.startswith('J'):
            return wordnet.ADJ
        elif treebank_tag.startswith('V'):
            return wordnet.VERB
        elif treebank_tag.startswith('N'):
            return wordnet.NOUN
        elif treebank_tag.startswith('R'):
            return wordnet.ADV
        else:
            return wordnet.NOUN

    text = re.sub(r'\.(?=[A-Z])', '. ', text)                                           # fix missing space after period
    text = re.sub(r'-\s', ' ', text)                                          # replace hyphen+space combos cleanly, optional refinement
    text = text.lower()                                                                 # lowercase
    tokens = word_tokenize(text)                                                        # tokenize
    tokens = [w for w in tokens if w not in punctuations and w not in stop_words]       # remove punctuation + stopwords
    tagged = pos_tag(tokens)
    tokens = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]           # pos + lemmatize
    # tokens = [lemmatizer.lemmatize(w) for w in tokens]                                # lemmatize
    return ' '.join(tokens)  

In [214]:
text_data['cleaned_summary'] = text_data['brief_summary'].apply(cleaned_text)

In [215]:
print("Data shape : ", text_data.shape)
print("-"*50)

print("\nFirst 5 records")
print(text_data[['brief_summary', 'cleaned_summary']].head())
print("-"*50)

Data shape :  (60337, 3)
--------------------------------------------------

First 5 records
                                       brief_summary  \
0  Breast cancer patients often have perioperativ...   
1  Many breast cancer patients experience psychol...   
2  Based on an American study by Scherer et al., ...   
3  Compare the effect of ropivacaine versus place...   
4  Phase 1 dose escalation and expansion study of...   

                                     cleaned_summary  
0  breast cancer patient often perioperative emot...  
1  many breast cancer patient experience psycholo...  
2  base american study scherer et al. hypothesize...  
3  compare effect ropivacaine versus placebo pect...  
4  phase 1 dose escalation expansion study clsp-1...  
--------------------------------------------------


#### Check the null value

In [216]:
print(text_data['cleaned_summary'].isnull().sum())
print((text_data['cleaned_summary'] == '').sum())

0
0


#### Checking the sample 5 random records:

In [217]:
sample_check = text_data.sample(5, random_state=41)
for idx, row in sample_check.iterrows():
    print("ORIGINAL:", row['brief_summary'])
    print("CLEANED:", row['cleaned_summary'])
    print("-" * 50)

ORIGINAL: This trial will assess chemosensitivity differences of the carotid bodies in individuals with T2DM, compared to healthy controls. During baseline and hyperinsulinemia.
CLEANED: trial assess chemosensitivity difference carotid body individual t2dm compare healthy control baseline hyperinsulinemia
--------------------------------------------------
ORIGINAL: Determination of incidence and prevalence of PTSD and other types of psychopathology (such as anxiety and affective disorders) after traumatic birth experiences and elucidation of salient risk factors in the local population sample- by prospective follow-up.
CLEANED: determination incidence prevalence ptsd type psychopathology anxiety affective disorder traumatic birth experience elucidation salient risk factor local population sample prospective follow-up
--------------------------------------------------
ORIGINAL: 1. To develop brief informational videos, Vidscrips, that can be sent to women following their mammogram to pr

#### save the text_data into csv file format

In [ ]:
text_data.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_5-Clinical_Trial_Disease_Category_Classification\data\preprocessed_data\cleaned_text_data.csv",index = False)
